# Part 5. Modelling and Validation
## Chronological evaluation, decision costs and a single sealed holdout

**Verified input:** Part 4 `run_20261115T093554Z`, derived from the Part 2 panel. Part 3 remains processing-method evidence, not an alternate modelling dataset.

**Protocol:** three expanding-window development folds, four-quarter purges, development-fitted preprocessing in each fold, average precision for model selection, an explicit FDIC-cost-based decision scenario, a Spark MLlib demonstration, and one final test evaluation. **Do not run Section 7 until model, threshold and costs have been sealed in Section 4.**

**Sequence:** 1. Environment/tests → 2. Upstream identity → 3. Frozen study design → 4. Development modelling/lock → 5. Development evidence → 6. Spark MLlib evidence → 7. Single test evaluation → 8. Read-only verification.

### 1. Source code, environment and integration tests

The source revision and pinned packages define the run. Tests cover temporal purges, metric arithmetic, threshold optimization and a real Spark MLlib model on independent synthetic records. All mandatory tests must execute without failure or skip before the FDIC analysis.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET

REPOSITORY = "https://github.com/Duongnb58/BFT65B_11230945_Final.git"
REPO_DIR = Path(tempfile.mkdtemp(prefix="bft65b_part5_", dir="/content")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)
required = (
    "requirements_part5.txt", "run_modelling_validation.py",
    "src/fdic_pipeline/model_policy.py", "src/fdic_pipeline/modelling_validation.py",
    "src/fdic_pipeline/feature_policy.py", "src/fdic_pipeline/feature_pipeline.py",
    "tests/test_model_policy.py", "tests/test_model_integration.py",
)
missing = [x for x in required if not (REPO_DIR / x).is_file()]
assert not missing, f"Missing Part 5 source files: {missing}"
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
java = subprocess.run(["java", "-version"], check=True, capture_output=True, text=True)
java_line = (java.stderr or java.stdout).splitlines()[0]
match = re.search(r'(?:version|openjdk)\s+"?(\d+)', java_line)
assert match and int(match.group(1)) >= 17, f"Java 17+ required: {java_line}"

subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "requirements_part5.txt")], check=True)
TEST_XML = Path("/content/bft65b_part5_tests.xml")
subprocess.run([
    sys.executable, "-m", "pytest", "-q", "-rs", "--junitxml", str(TEST_XML),
    str(REPO_DIR / "tests/test_model_policy.py"),
    str(REPO_DIR / "tests/test_model_integration.py"),
], cwd=REPO_DIR, check=True)
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites and sum(int(x.get("skipped", 0)) for x in suites) == 0
assert sum(int(x.get("errors", 0)) + int(x.get("failures", 0)) for x in suites) == 0
sys.path.insert(0, str(REPO_DIR / "src"))
print("Part 5 tests: PASSED; skipped: 0")
print("Java:", java_line)
print("Git revision:", GIT_COMMIT)

### 2. Input contract: Part 4 and upstream evidence

This stage uses the exact Part 4 run. Source identity is checked through manifests and SHA-256 metadata. No holdout predictor or outcome values are loaded in this section. Part 3 is checked for methodological provenance only.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from fdic_pipeline.modelling_validation import _validate_part4, _verify_part3

DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945_Final")
ACQUISITION_DIR = DRIVE_ROOT / "fdic_1984Q1_2026Q2"
PART2_DIR = DRIVE_ROOT / "data_quality_panel/run_20261114T175314Z"
PART3_DIR = DRIVE_ROOT / "scalable_processing/run_20261114T190627Z"
PART4_DIR = DRIVE_ROOT / "feature_pipeline/run_20261115T093554Z"
SOURCE_MANIFEST = _validate_part4(ACQUISITION_DIR, PART2_DIR, PART4_DIR)
PART3_REFERENCE = _verify_part3(PART3_DIR, PART2_DIR)
print("Part 1–4 source contract: PASSED")
print("Part 4 run:", PART4_DIR.name)
print("Development partitions:", len(SOURCE_MANIFEST["development_partitions"]))
print("Test partitions (metadata only):", len(SOURCE_MANIFEST["test_holdout_partitions"]))
print("Part 3 provenance:", PART3_REFERENCE["run_id"])
print("Holdout contents inspected: NO")

### 3. Frozen validation, candidate models and alarm-cost policy

All validation windows precede 2007. A four-quarter purge prevents training outcomes from overlapping a validation window. The three models are a prevalence baseline, penalized logistic regression and histogram gradient boosting. The selected model maximizes the mean fold average precision; the alarm threshold minimizes a development-only decision-loss scenario. The assumed false-alarm expense is **not** an observed FDIC supervision cost.

In [ ]:
from datetime import datetime, timezone
from fdic_pipeline.model_policy import (
    FOLDS, MODEL_NAMES, FALSE_ALARM_ASSET_RATE, SEED, validate_folds,
)
validate_folds()
RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
OUTPUT_DIR = DRIVE_ROOT / "modelling_validation" / RUN_ID
assert not OUTPUT_DIR.exists(), f"Run already exists: {OUTPUT_DIR}"
print("Part 5 run:", RUN_ID)
print("Models:", MODEL_NAMES)
for fold in FOLDS:
    print(fold.name, "training through", fold.training_end,
          "validation", fold.validation_start, "to", fold.validation_end)
print("False-alarm review-cost proxy / median development assets:", FALSE_ALARM_ASSET_RATE)
print("Reproducible seed:", SEED)
print("Output:", OUTPUT_DIR)

### 4. Development-only training, model selection and sealed threshold

This is the computational stage. Every fold fits a fresh preprocessing pipeline using only its training quarters. Model selection uses validation average precision; FDIC `COST` before development end and an explicitly assumed supervisory-review proxy determine the cost-sensitive alarm threshold from validation predictions. A local Spark MLlib demonstration uses development data only. **No test files are read.** The stage ends by sealing `selection.lock.json`; subsequent test evaluation cannot change the selected model or threshold.

In [ ]:
TRAIN_CMD = [
    sys.executable, str(REPO_DIR / "run_modelling_validation.py"), "train",
    "--acquisition-dir", str(ACQUISITION_DIR),
    "--part2-dir", str(PART2_DIR),
    "--part4-dir", str(PART4_DIR),
    "--part3-dir", str(PART3_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--git-commit", GIT_COMMIT,
]
subprocess.run(TRAIN_CMD, cwd=REPO_DIR, check=True)
import json
LOCK = json.loads((OUTPUT_DIR / "selection.lock.json").read_text(encoding="utf-8"))
assert LOCK["state"] == "DEVELOPMENT_SELECTION_SEALED"
assert not (OUTPUT_DIR / "test_attempt.json").exists()
print("Development selection: SEALED")
print("Chosen model:", LOCK["selected_model"])
print("Locked alarm threshold:", LOCK["threshold"])
print("Holdout accessed: NO")

### 5. Review development cross-validation and cost scenario

The tables below are derived only from historical validation quarters, not the test holdout. Average precision is prioritized because the positive class is rare. The alarm threshold is fitted on the selected model's out-of-fold predictions; sensitivity scenarios vary the assumed review expense while holding observed FDIC failure costs fixed.

In [ ]:
import csv
from IPython.display import display, Markdown, Image

def load_csv(name):
    with (OUTPUT_DIR / name).open(encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))

def show_table(title, fields, records):
    display(Markdown("**" + title + "**"))
    display(Markdown(" | ".join(fields) + "\n" + " | ".join("---" for _ in fields)
                     + "\n" + "\n".join(" | ".join(str(r.get(k, "")) for k in fields)
                                             for r in records)))

show_table("Model ranking on three purged development folds",
           ["model", "mean_fold_ap", "mean_fold_roc_auc", "selected"],
           load_csv("development_model_comparison.csv"))
POLICY = json.loads((OUTPUT_DIR / "threshold_policy.json").read_text())
COSTS = POLICY["costs"]
print("FDIC-positive failure COST records, development only:",
      COSTS["usable_development_failure_cost_records"])
print("Missed-failure cost proxy (USD thousands):", COSTS["missed_failure_cost_thousands_usd"])
print("Assumed false-alarm review cost (USD thousands):", COSTS["false_alarm_cost_thousands_usd"])
show_table("Alarm-cost sensitivity: development out-of-fold data",
           ["false_alarm_multiplier", "threshold", "tp", "fp", "fn", "estimated_cost_thousands_usd"],
           POLICY["scenarios"])
display(Markdown("**Development out-of-fold precision–recall**"))
display(Image(filename=str(OUTPUT_DIR / "figures/development_pr.png"), width=690))
display(Markdown("**Average precision by historical fold**"))
display(Image(filename=str(OUTPUT_DIR / "figures/development_fold_ap.png"), width=690))

### 6. Spark MLlib: development-only demonstration

The demonstration fits a Spark-native Imputer, VectorAssembler and logistic regression on a bounded development subset using the first chronological fold. Its validation is development-only and does not compete in the final model ranking; it demonstrates the MLlib pipeline and a working local JVM.

In [ ]:
SPARK_DEMO = json.loads((OUTPUT_DIR / "spark_mllib_demo.json").read_text())
assert SPARK_DEMO["engine"] == "Spark MLlib, local[2]"
assert SPARK_DEMO["training_source"] == "development only"
assert SPARK_DEMO["validation_metrics"]["observations"] > 0
print("Spark MLlib demonstration: PASSED")
print("Train/validation rows:", SPARK_DEMO["training_rows"], SPARK_DEMO["validation_rows"])
print("Validation average precision:", SPARK_DEMO["validation_metrics"]["average_precision"])
print("Spark predictors:", SPARK_DEMO["predictors"])
print("Spark is local[2]; no multinode throughput inference")

### 7. Final holdout evaluation — execute exactly once

Only now does the program open test Parquet files. It loads the **already sealed** model and threshold, evaluates the selected model and the predefined prevalence baseline, and saves predictions and metrics. A `test_attempt.json` marker is written before test data are read. Running this section again on the same run is refused, even after an interrupted evaluation. No model, preprocessing or threshold is refitted here.

In [ ]:
EVALUATE_CMD = [
    sys.executable, str(REPO_DIR / "run_modelling_validation.py"), "evaluate",
    "--acquisition-dir", str(ACQUISITION_DIR),
    "--part2-dir", str(PART2_DIR),
    "--part4-dir", str(PART4_DIR),
    "--output-dir", str(OUTPUT_DIR),
]
subprocess.run(EVALUATE_CMD, cwd=REPO_DIR, check=True)
FINAL = json.loads((OUTPUT_DIR / "part5_validation.json").read_text())
assert FINAL["success"] is True and FINAL["holdout_evaluated_once"] is True
print("Final test evaluation: COMPLETED ONCE")
print("Test observations:", FINAL["test_rows"])
print("Test positives:", FINAL["test_positives"])
print("Selected model:", FINAL["model"])
print("Locked threshold:", FINAL["threshold"])

### 8. Independent evidence verification and Part 6 handoff

The verification command checks source lineage and the sealed development choice. It recalculates test metrics **from already saved test predictions**, not by reopening the test dataset or scoring the model again. The results must be preserved for the final report; do not rerun training or the held-out evaluation to improve reported performance.

In [ ]:
VERIFY_CMD = [
    sys.executable, str(REPO_DIR / "run_modelling_validation.py"), "verify",
    "--acquisition-dir", str(ACQUISITION_DIR),
    "--part2-dir", str(PART2_DIR),
    "--part4-dir", str(PART4_DIR),
    "--output-dir", str(OUTPUT_DIR),
]
subprocess.run(VERIFY_CMD, cwd=REPO_DIR, check=True)
show_table("Single holdout evaluation (no post-test model changes)",
           ["model", "average_precision", "roc_auc", "brier_score", "top_5pct_recall",
            "threshold", "tp", "fp", "fn", "estimated_cost_thousands_usd"],
           load_csv("test_metrics.csv"))
display(Markdown("**Final holdout precision–recall**"))
display(Image(filename=str(OUTPUT_DIR / "figures/test_pr.png"), width=690))
display(Markdown("**Frozen-threshold confusion matrix**"))
display(Image(filename=str(OUTPUT_DIR / "figures/test_confusion.png"), width=620))
print("Part 5 evidence: PASSED")
print("Run:", RUN_ID)
print("Git revision:", GIT_COMMIT)
print("Final development lock:", OUTPUT_DIR / "selection.lock.json")
print("Single test evidence:", OUTPUT_DIR / "evaluation_manifest.json")

## Reproducibility and interpretation

Only the selected model's locked configuration may be treated as the final system. FDIC resolution `COST` is an observed proxy for missed failure; the false-alarm expense is an assumption based on development bank assets, not an observed regulator expenditure. Decision costs are evaluated **per bank-quarter alarm**, so repeated warnings about one institution can contribute multiple decisions. The 2008–2014 crisis holdout provides a stress-period assessment, not a universal out-of-time guarantee. Preserve the completed Part 5 run for the final report and Part 6 conclusions.